# Chapter 4: LlamaIndex + ChromaDB — Retrieval-Augmented Generation (RAG)

### Why This Matters

An LLM's knowledge is frozen at its training cutoff and knows nothing about your company's private documents — policies, reports, product manuals. **Retrieval-Augmented Generation (RAG)** solves this by retrieving the most *relevant* pieces of your own documents at question time and inserting them into the prompt as context, letting the model answer grounded in your actual, current, private information rather than only its general training. This is, by a wide margin, the most common and most immediately valuable pattern for deploying LLMs inside a real business today.

### 4.1 The RAG Pipeline, Conceptually

```
documents -> parsing -> chunks -> embeddings -> vector/index storage
user question -> retrieval (find the most relevant chunks) -> context -> LLM -> grounded answer
```

Two distinct phases matter here, and confusing them is a common source of RAG systems that perform poorly: **indexing** (done once, or whenever documents change — parse your documents, split them into manageable chunks, convert each chunk to an embedding vector via Chapter 2's `SentenceTransformer`-style models, and store them) and **retrieval** (done on every user question — embed the question itself, find the stored chunks whose embeddings are most similar, and feed only those relevant chunks to the LLM as context). The retrieval step is precisely what connects a general-purpose LLM to your specific, private knowledge.

### 4.2 ChromaDB — The Vector Store

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("demo")
collection.add(
    ids=["1", "2"],
    documents=["Refunds allowed within 30 days", "Premium plan includes priority support"],
    metadatas=[{"topic": "returns"}, {"topic": "support"}]
)
print(collection.count())     # 2

result = collection.query(query_texts=["How long do I have to return an item?"], n_results=2)
print(result["documents"])

A **vector database** like ChromaDB is purpose-built for exactly one specialized job: storing high-dimensional embedding vectors and efficiently finding the nearest neighbors to a query vector (this is called **similarity search**, typically measured via **cosine similarity** — how closely two vectors point in the same direction, regardless of their magnitude). Notice `collection.query(query_texts=[...])` — ChromaDB handles embedding the query text automatically behind the scenes (using a default or configured embedding model) and returns the documents whose *meaning* is closest, not documents that share exact keywords, which is exactly what happened above: the query about "return an item" correctly matched the stored document about "refunds," despite sharing no exact word beyond "item"/nothing at all.

⚠️ **A vector database is not a replacement for a transactional database (Volume 10's MySQL/MongoDB) — it solves a fundamentally different problem.** Use a vector database for semantic search and RAG retrieval; keep your actual transactional business data (orders, customers, inventory) in a proper relational or document database, and treat the vector store as a specialized index layered on top of your real documents, not as your system of record.

`metadatas` (attached to each stored document) matters enormously in real deployments — it lets you filter retrieval by department, document date, access level, or source, so a query can be restricted to (for example) only documents a specific user is authorized to see, or only policies updated within the last year.

### 4.3 LlamaIndex — Purpose-Built for the Full RAG Workflow

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import chromadb

# 1. Load your documents (PDFs, text, CSVs, etc.)
documents = SimpleDirectoryReader("company_docs/").load_data()

# 2. Create/connect to a persistent Chroma vector store
chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("business_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 3. Build the index (this runs the parsing -> chunking -> embedding -> storage pipeline for you)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(documents, vector_store=vector_store)

# 4. Query it
query_engine = index.as_query_engine()
response = query_engine.query("What was our Q3 revenue growth in the North region?")
print(response)

While the previous section showed ChromaDB's storage/retrieval mechanics directly, LlamaIndex automates the *entire surrounding pipeline* — document loading from a folder of mixed file types, intelligent chunking (splitting long documents at sensible boundaries rather than arbitrary character counts), embedding generation, and finally wiring retrieval into an LLM call, all through the higher-level `query_engine.query()` interface. This is precisely the tool of choice when you want to stand up an internal "ask questions about our documents" assistant with minimal custom plumbing.

### 4.4 What Actually Determines RAG Quality

The source material's exercises point at exactly the right questions to ask when *designing* (not just implementing) a RAG system, because the plumbing above is the easy part — getting good answers depends on decisions upstream of any code:

- **Chunk size strategy** — chunks too small lose context; chunks too large dilute retrieval precision and waste the LLM's limited context window. There's no universal right answer; it depends on document structure (a policy document with short discrete sections behaves differently than a long narrative report).
- **Metadata design** — as noted above, attaching source, date, section, and access-level metadata to every chunk is what makes filtered, auditable, access-controlled retrieval possible, rather than an unstructured pile of text fragments.
- **Retrieval evaluation** — metrics like **precision@k** ("of the top k retrieved chunks, how many were actually relevant") and **recall@k** ("of all truly relevant chunks, how many did we retrieve in the top k") let you measure and improve retrieval quality directly, rather than only judging the final generated answer, which conflates retrieval quality with the LLM's own writing quality.

---